# Cross-sectional Diebold-Mariano tests

This notebook discovers every model folder in `Results` containing a same-named main prediction file such as `LSTM_AIC/LSTM_AIC.csv`, loads realised volatility from `Data/<ticker>.csv` (same parser and duplicate policy as the model notebooks), and compares models pairwise.

Following Zhang et al. (2024), Appendix C, Equation (31), and Gu, Kelly and Xiu (2020), the loss differential is averaged across the assets available on each date before the Diebold-Mariano statistic is calculated through time:

$$d_t^{(a-b)} = \frac{1}{N_t} \sum_{i=1}^{N_t} \left[L(e_{i,t}^{(a)}) - L(e_{i,t}^{(b)})\right].$$

The default loss is QLIKE. The long-run variance uses the Newey-West (Bartlett) estimator with $\lfloor 4(T/100)^{2/9} \rfloor$ lags; p-values are two-sided from the standard normal. A negative statistic means the row (or first) model has lower loss.

The notebook reports (i) all pairwise tests, (ii) every model against the reference model `REFERENCE_MODEL` (LSTM_AIC) for QLIKE and MSE, and (iii) the reference comparisons within the pre-specified sub-periods. Robustness runs (folders ending in `_refit<N>m`, `_aic<N>y`, `_resel<N>m` or `_seed<N>`) are excluded unless `INCLUDE_ROBUSTNESS = True` or they are listed in `MODELS`.

MSE is computed on variances (squared volatilities), which keeps it robust to noise in the realised measure; `mse_log` (squared error of log volatility) is a check that the missing log-to-level bias correction does not drive the rankings.

The final sections report (iv) the **noise floor**: the same model (`LSTM_21`) estimated with two independent seed sets, which shows how large a difference seed noise alone produces; (v) **per-stock DM tests** against the reference model, counted as significant wins and losses; and (vi) an **oracle**: for each stock, the fixed lookback with the lowest full-sample QLIKE in hindsight, and the share of the gap between `LSTM_21` and the oracle that each selection rule captures.

In [ ]:
from pathlib import Path
from itertools import combinations
from math import erfc, sqrt
import re

import numpy as np
import pandas as pd
from IPython.display import display

# User settings
MODELS = ["LSTM_10", "LSTM_21", "LSTM_42", "LSTM_63", "LSTM_VAL", "LSTM_AIC"]  # None: every non-robustness folder
INCLUDE_ROBUSTNESS = False        # include robustness folders (*_refit<N>m, *_aic<N>y, *_resel<N>m, *_seed<N>)
REFERENCE_MODEL = "LSTM_AIC"      # model compared against every other model
LOSS = "qlike"                    # "qlike", "mse" (on variances) or "mse_log" for the pairwise table
GRID = [10, 21, 42, 63]           # fixed lookbacks, for the oracle
NOISE_FLOOR_PAIR = ("LSTM_21", "LSTM_21_seed1")  # same model, independent seeds
VALUES_ARE_VOLATILITY = True      # True: QLIKE is applied to squared values (variances)
ESTIMATE_TYPE = "QMLE-Trade"      # Matches the model configs
VALUE_COLUMN = "Volatility"
DUPLICATE_POLICY = "drop"         # must match duplicate_policy in the model notebooks
HAC_LAGS = None                   # None: automatic Newey-West lag rule
ALPHA = 0.05

# Pre-specified sub-periods (test dates). "Other years" is every test date
# outside the three named periods.
PERIODS = {
    "Full sample": None,
    "Early sample (2004-2006)": [("2004-01-01", "2006-12-31")],
    "Financial crisis (2008-2009)": [("2008-01-01", "2009-12-31")],
    "COVID-19 (2020)": [("2020-01-01", "2020-12-31")],
    "Other years": "complement",
}

pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

In [ ]:
def find_project_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "Results").is_dir() and (candidate / "Data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate a project root containing Results/ and Data/.")


def natural_key(name: str):
    return [int(part) if part.isdigit() else part.lower() for part in re.split(r"(\d+)", name)]


def discover_models(results_dir: Path, models=None, include_robustness=False):
    """Every Results/<name>/<name>.csv. Robustness runs (names ending in
    _refit<N>m, _aic<N>y, _resel<N>m or _seed<N>) are excluded unless
    requested or listed explicitly."""
    found = {}
    for folder in results_dir.iterdir():
        main_file = folder / f"{folder.name}.csv"
        if folder.is_dir() and main_file.is_file():
            found[folder.name] = main_file
    if models is not None:
        missing = [m for m in models if m not in found]
        if missing:
            print(f"Not yet run (skipped): {missing}")
        found = {m: found[m] for m in models if m in found}
    elif not include_robustness:
        found = {k: v for k, v in found.items()
                 if not re.search(r"_(refit\d+m|aic\d+y|resel\d+m|seed\d+)", k)}
    return dict(sorted(found.items(), key=lambda item: natural_key(item[0])))


def parse_source_dates(values):
    """Same date parser as NN/NN.ipynb."""
    text = values.astype(str).str.strip()
    slash_mask = text.str.contains("/", regex=False)
    parsed = pd.Series(pd.NaT, index=text.index, dtype="datetime64[ns]")
    parsed.loc[~slash_mask] = pd.to_datetime(text.loc[~slash_mask], format="%Y-%m-%d", errors="raise")
    parsed.loc[slash_mask] = pd.to_datetime(text.loc[slash_mask], dayfirst=True, errors="raise")
    return parsed


def load_prediction_file(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path)
    if "date" not in frame.columns:
        raise ValueError(f"Missing date column in {path}")
    frame["date"] = pd.to_datetime(frame["date"], errors="raise")
    if frame["date"].duplicated().any():
        raise ValueError(f"Duplicate dates in {path}")
    frame = frame.set_index("date").sort_index()
    return frame.apply(pd.to_numeric, errors="coerce")


def load_actuals(tickers, data_dir: Path) -> pd.DataFrame:
    """Realised volatility, constructed exactly as in the model notebooks:
    same estimate type, same date parser, same duplicate policy."""
    series = []
    for ticker in tickers:
        path = data_dir / f"{ticker}.csv"
        if not path.is_file():
            raise FileNotFoundError(f"No realised-volatility file for {ticker}: {path}")
        frame = pd.read_csv(path)
        frame = frame.loc[frame["Type"].eq(ESTIMATE_TYPE), ["Date", VALUE_COLUMN]].copy()
        if frame.empty:
            raise ValueError(f"No {ESTIMATE_TYPE!r} observations in {path}")
        frame["Date"] = parse_source_dates(frame["Date"])
        duplicated = frame["Date"].duplicated(keep=False)
        if duplicated.any():
            dates = sorted(frame.loc[duplicated, "Date"].dt.date.unique())
            if DUPLICATE_POLICY == "drop":
                frame = frame.loc[~duplicated]
                action = "treated as missing"
            elif DUPLICATE_POLICY == "first":
                frame = frame.drop_duplicates("Date", keep="first")
                action = "first observation kept"
            else:
                raise ValueError("DUPLICATE_POLICY must be 'drop' or 'first'")
            print(f"Data note - {ticker}: repeated observations on {len(dates)} date(s) {action}: {dates}")
        item = pd.to_numeric(frame[VALUE_COLUMN], errors="coerce")
        item.index = frame["Date"]
        item.name = ticker
        series.append(item)
    return pd.concat(series, axis=1, sort=True)

def period_mask(dates, period):
    """Boolean mask over a DatetimeIndex for one entry of PERIODS."""
    dates = pd.DatetimeIndex(dates)
    if period is None:
        return np.ones(len(dates), dtype=bool)
    if isinstance(period, str) and period == "complement":
        named = np.zeros(len(dates), dtype=bool)
        for other in PERIODS.values():
            if isinstance(other, list):
                named |= period_mask(dates, other)
        return ~named
    mask = np.zeros(len(dates), dtype=bool)
    for start, end in period:
        mask |= (dates >= pd.Timestamp(start)) & (dates <= pd.Timestamp(end))
    return mask


ROOT = find_project_root(Path.cwd())
RESULTS_DIR = ROOT / "Results"
DATA_DIR = ROOT / "Data"

model_files = discover_models(RESULTS_DIR, MODELS, INCLUDE_ROBUSTNESS)
if len(model_files) < 2:
    raise ValueError(f"Need at least two model result folders; found {list(model_files)}")

print(f"Project root: {ROOT}")
print(f"Discovered {len(model_files)} models: {', '.join(model_files)}")

In [ ]:
def loss_values(actual, predicted, loss=LOSS):
    actual = np.asarray(actual, dtype=float)
    predicted = np.asarray(predicted, dtype=float)
    if loss == "mse":
        # Squared error of the variance forecast.
        actual_var = np.square(actual) if VALUES_ARE_VOLATILITY else actual
        predicted_var = np.square(predicted) if VALUES_ARE_VOLATILITY else predicted
        return np.square(actual_var - predicted_var)
    if loss == "mse_log":
        # Squared error of log volatility: unaffected by the level bias of exp(log forecast).
        if np.any(actual <= 0) or np.any(predicted <= 0):
            raise ValueError("mse_log requires strictly positive values.")
        return np.square(np.log(actual) - np.log(predicted))
    if loss == "qlike":
        actual_scale = np.square(actual) if VALUES_ARE_VOLATILITY else actual
        predicted_scale = np.square(predicted) if VALUES_ARE_VOLATILITY else predicted
        if np.any(actual_scale <= 0) or np.any(predicted_scale <= 0):
            raise ValueError("QLIKE requires strictly positive realised and predicted variance.")
        ratio = actual_scale / predicted_scale
        return ratio - np.log(ratio) - 1.0
    raise ValueError("LOSS must be 'qlike', 'mse' or 'mse_log'.")

In [ ]:
predictions = {name: load_prediction_file(path) for name, path in model_files.items()}
all_tickers = sorted(set().union(*(frame.columns for frame in predictions.values())))
actuals = load_actuals(all_tickers, DATA_DIR)

coverage_rows = []
for model, predicted in predictions.items():
    dates = predicted.index.intersection(actuals.index)
    tickers = predicted.columns.intersection(actuals.columns)
    p = predicted.loc[dates, tickers].to_numpy(dtype=float)
    y = actuals.loc[dates, tickers].to_numpy(dtype=float)
    valid = np.isfinite(p) & np.isfinite(y)
    if LOSS in ("qlike", "mse_log"):
        valid &= (p > 0) & (y > 0)
    losses = loss_values(y[valid], p[valid], LOSS)
    coverage_rows.append({
        "model": model,
        "tickers": len(tickers),
        "observations": int(valid.sum()),
        f"mean_{LOSS}": float(losses.mean()),
        "first_prediction": predicted[tickers].stack().index.get_level_values(0).min(),
        "last_prediction": predicted[tickers].stack().index.get_level_values(0).max(),
    })

coverage = pd.DataFrame(coverage_rows).set_index("model")
display(coverage)

In [ ]:
def automatic_hac_lags(sample_size: int) -> int:
    # Common Newey-West rule of thumb; bounded so it is valid for short samples.
    return min(sample_size - 1, int(np.floor(4 * (sample_size / 100) ** (2 / 9))))


def newey_west_long_run_variance(values, lags: int) -> float:
    values = np.asarray(values, dtype=float)
    centered = values - values.mean()
    sample_size = centered.size
    long_run_variance = float(centered @ centered / sample_size)
    for lag in range(1, lags + 1):
        weight = 1.0 - lag / (lags + 1.0)  # Bartlett kernel
        autocovariance = float(centered[lag:] @ centered[:-lag] / sample_size)
        long_run_variance += 2.0 * weight * autocovariance
    return max(long_run_variance, 0.0)


def cross_sectional_dm(model_a: str, model_b: str, loss=LOSS, hac_lags=HAC_LAGS, period=None):
    pred_a = predictions[model_a]
    pred_b = predictions[model_b]
    dates = pred_a.index.intersection(pred_b.index).intersection(actuals.index)
    dates = dates[period_mask(dates, period)]
    tickers = pred_a.columns.intersection(pred_b.columns).intersection(actuals.columns)
    if dates.empty or tickers.empty:
        raise ValueError(f"No common date/asset observations for {model_a} and {model_b}.")

    a = pred_a.loc[dates, tickers].to_numpy(dtype=float)
    b = pred_b.loc[dates, tickers].to_numpy(dtype=float)
    y = actuals.loc[dates, tickers].to_numpy(dtype=float)
    valid = np.isfinite(a) & np.isfinite(b) & np.isfinite(y)
    if loss in ("qlike", "mse_log"):
        valid &= (a > 0) & (b > 0) & (y > 0)

    loss_a = np.full(a.shape, np.nan)
    loss_b = np.full(b.shape, np.nan)
    loss_a[valid] = loss_values(y[valid], a[valid], loss)
    loss_b[valid] = loss_values(y[valid], b[valid], loss)
    differential = loss_a - loss_b

    assets_per_date = valid.sum(axis=1)
    daily_differential = np.divide(
        np.nansum(differential, axis=1),
        assets_per_date,
        out=np.full(len(dates), np.nan),
        where=assets_per_date > 0,
    )
    keep = np.isfinite(daily_differential)
    d_t = daily_differential[keep]
    n_t = assets_per_date[keep]
    if d_t.size < 2:
        raise ValueError(f"Fewer than two usable dates for {model_a} and {model_b}.")

    lags = automatic_hac_lags(d_t.size) if hac_lags is None else int(hac_lags)
    if not 0 <= lags < d_t.size:
        raise ValueError(f"HAC lags must be between 0 and {d_t.size - 1}.")
    long_run_variance = newey_west_long_run_variance(d_t, lags)
    standard_error = sqrt(long_run_variance / d_t.size)
    statistic = float(d_t.mean() / standard_error) if standard_error > 0 else np.nan
    p_value = float(erfc(abs(statistic) / sqrt(2))) if np.isfinite(statistic) else np.nan
    winner = model_a if d_t.mean() < 0 else model_b if d_t.mean() > 0 else "tie"

    return {
        "model_a": model_a,
        "model_b": model_b,
        "loss": loss,
        "mean_loss_difference_a_minus_b": float(d_t.mean()),
        "dm_statistic": statistic,
        "p_value_two_sided": p_value,
        "significant_5pct": bool(p_value < ALPHA),
        "lower_loss_model": winner,
        "dates": int(d_t.size),
        "min_assets_per_date": int(n_t.min()),
        "max_assets_per_date": int(n_t.max()),
        "mean_assets_per_date": float(n_t.mean()),
        "hac_lags": lags,
    }

In [ ]:
models = list(model_files)
unique_results = []
directional_results = []

for model_a, model_b in combinations(models, 2):
    result = cross_sectional_dm(model_a, model_b)
    unique_results.append(result)
    directional_results.append(result)

    reverse = result.copy()
    reverse["model_a"], reverse["model_b"] = model_b, model_a
    reverse["mean_loss_difference_a_minus_b"] *= -1
    reverse["dm_statistic"] *= -1
    directional_results.append(reverse)

unique_results = pd.DataFrame(unique_results)
directional_results = pd.DataFrame(directional_results).sort_values(["model_a", "model_b"]).reset_index(drop=True)

print(f"Completed {len(unique_results)} unique tests and {len(directional_results)} directional comparisons.")

In [ ]:
def significance_stars(p_value):
    if p_value < 0.01:
        return "***"
    if p_value < 0.05:
        return "**"
    if p_value < 0.10:
        return "*"
    return ""


# Upper-right (inverse right-angle) triangle. Each entry is DM statistic [two-sided p-value].
triangle = pd.DataFrame("", index=models, columns=models)
for model in models:
    triangle.loc[model, model] = "—"
for _, result in unique_results.iterrows():
    triangle.loc[result.model_a, result.model_b] = (
        f"{result.dm_statistic:.3f}{significance_stars(result.p_value_two_sided)} "
        f"[p={result.p_value_two_sided:.4f}]"
    )

triangle_style = (
    triangle.style
    .set_caption(
        f"Cross-sectional DM tests using {LOSS.upper()} loss — row model minus column model"
    )
    .set_properties(**{"text-align": "center", "padding": "8px 12px"})
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px"), ("margin-bottom", "8px")]},
        {"selector": "th", "props": [("background-color", "#16324F"), ("color", "white"), ("padding", "8px 12px")]},
        {"selector": "td", "props": [("border-bottom", "1px solid #D9E2EC")]},
    ])
)
display(triangle_style)
print("Negative DM: row model has lower loss. Positive DM: column model has lower loss.")
print("Significance: * p<0.10, ** p<0.05, *** p<0.01.")

In [ ]:
# All ordered comparisons. For k models, this table has k(k-1) rows.
columns = [
    "model_a", "model_b", "dm_statistic", "p_value_two_sided",
    "mean_loss_difference_a_minus_b", "lower_loss_model",
    "significant_5pct", "dates", "mean_assets_per_date", "hac_lags",
]
display(
    directional_results[columns].style
    .format({
        "dm_statistic": "{:.3f}",
        "p_value_two_sided": "{:.4f}",
        "mean_loss_difference_a_minus_b": "{:.8f}",
        "mean_assets_per_date": "{:.2f}",
    })
    .hide(axis="index")
    .set_caption("All directional comparisons")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "14px"), ("margin-bottom", "8px")]},
        {"selector": "th", "props": [("background-color", "#16324F"), ("color", "white"), ("padding", "7px")]},
        {"selector": "td", "props": [("padding", "6px 8px"), ("border-bottom", "1px solid #D9E2EC")]},
    ])
)

## Every model against the reference model

Rows report `REFERENCE_MODEL` minus the column model. A negative statistic means the reference model has lower loss.

In [ ]:
if REFERENCE_MODEL not in model_files:
    print(f"Reference model {REFERENCE_MODEL} not found; skipping reference tables.")
    reference_table = pd.DataFrame()
else:
    rows = []
    for other in [m for m in models if m != REFERENCE_MODEL]:
        for loss in ["qlike", "mse", "mse_log"]:
            result = cross_sectional_dm(REFERENCE_MODEL, other, loss=loss)
            rows.append({
                "comparison": f"{REFERENCE_MODEL} vs {other}",
                "loss": loss.upper(),
                "dm_statistic": result["dm_statistic"],
                "p_value": result["p_value_two_sided"],
                "stars": significance_stars(result["p_value_two_sided"]),
                "mean_loss_difference": result["mean_loss_difference_a_minus_b"],
                "dates": result["dates"],
                "hac_lags": result["hac_lags"],
            })
    reference_table = pd.DataFrame(rows)
    display(reference_table.pivot(index="comparison", columns="loss", values=["dm_statistic", "p_value"]))

## Reference comparisons by sub-period (QLIKE)

In [ ]:
if REFERENCE_MODEL in model_files:
    rows = []
    for period_name, period in PERIODS.items():
        for other in [m for m in models if m != REFERENCE_MODEL]:
            try:
                result = cross_sectional_dm(REFERENCE_MODEL, other, loss="qlike", period=period)
            except ValueError as error:
                print(f"{period_name}, {other}: skipped ({error})")
                continue
            rows.append({
                "period": period_name,
                "other_model": other,
                "cell": f"{result['dm_statistic']:.2f}{significance_stars(result['p_value_two_sided'])}",
                "dm_statistic": result["dm_statistic"],
                "p_value": result["p_value_two_sided"],
                "dates": result["dates"],
            })
    period_table = pd.DataFrame(rows)
    display(period_table.pivot(index="period", columns="other_model", values="cell")
            .reindex(list(PERIODS)))

    output_dir = RESULTS_DIR / "Analysis" / "output"
    output_dir.mkdir(parents=True, exist_ok=True)
    unique_results.to_csv(output_dir / f"dm_pairwise_{LOSS}.csv", index=False)
    reference_table.to_csv(output_dir / "dm_reference.csv", index=False)
    period_table.to_csv(output_dir / "dm_reference_by_period.csv", index=False)
    print(f"Saved DM tables to {output_dir}")

## Noise floor: the same model with independent seeds

If `LSTM_21` and `LSTM_21_seed1` differ by as much as the lookback comparisons above, seed noise rather than the lookback explains those differences.

In [ ]:
noise_files = discover_models(RESULTS_DIR, list(NOISE_FLOOR_PAIR), include_robustness=True)
if len(noise_files) == 2:
    for name, path in noise_files.items():
        predictions.setdefault(name, load_prediction_file(path))
    rows = []
    for loss in ["qlike", "mse", "mse_log"]:
        result = cross_sectional_dm(*NOISE_FLOOR_PAIR, loss=loss)
        rows.append({"loss": loss.upper(),
                     "mean_loss_difference": result["mean_loss_difference_a_minus_b"],
                     "dm_statistic": result["dm_statistic"],
                     "p_value": result["p_value_two_sided"]})
    noise_floor = pd.DataFrame(rows)
    display(noise_floor)
    noise_floor.to_csv(RESULTS_DIR / "Analysis" / "output" / "dm_noise_floor.csv", index=False)
else:
    print(f"Noise-floor run not found yet: need both of {NOISE_FLOOR_PAIR}.")

## Per-stock DM tests against the reference model

For each stock, a DM test on that stock's daily loss differential (same Newey-West settings). The table counts stocks where the reference model is significantly better or worse at 5%.

In [ ]:
def per_stock_dm(model_a, model_b, loss="qlike", hac_lags=HAC_LAGS):
    pred_a, pred_b = predictions[model_a], predictions[model_b]
    dates = pred_a.index.intersection(pred_b.index).intersection(actuals.index)
    tickers = pred_a.columns.intersection(pred_b.columns).intersection(actuals.columns)
    rows = []
    for ticker in tickers:
        a = pred_a.loc[dates, ticker].to_numpy(dtype=float)
        b = pred_b.loc[dates, ticker].to_numpy(dtype=float)
        y = actuals.loc[dates, ticker].to_numpy(dtype=float)
        valid = np.isfinite(a) & np.isfinite(b) & np.isfinite(y) & (a > 0) & (b > 0) & (y > 0)
        d = loss_values(y[valid], a[valid], loss) - loss_values(y[valid], b[valid], loss)
        if d.size < 2:
            continue
        lags = automatic_hac_lags(d.size) if hac_lags is None else int(hac_lags)
        se = sqrt(newey_west_long_run_variance(d, lags) / d.size)
        stat = float(d.mean() / se) if se > 0 else np.nan
        p = float(erfc(abs(stat) / sqrt(2))) if np.isfinite(stat) else np.nan
        rows.append({"ticker": ticker, "dm_statistic": stat, "p_value": p})
    return pd.DataFrame(rows)


if REFERENCE_MODEL in predictions:
    count_rows, per_stock_all = [], []
    for other in [m for m in models if m != REFERENCE_MODEL]:
        for loss in ["qlike", "mse"]:
            table = per_stock_dm(REFERENCE_MODEL, other, loss)
            table["comparison"], table["loss"] = f"{REFERENCE_MODEL} vs {other}", loss.upper()
            per_stock_all.append(table)
            count_rows.append({
                "comparison": f"{REFERENCE_MODEL} vs {other}", "loss": loss.upper(),
                "reference significantly better": int(((table.dm_statistic < 0) & (table.p_value < ALPHA)).sum()),
                "reference significantly worse": int(((table.dm_statistic > 0) & (table.p_value < ALPHA)).sum()),
                "stocks": int(len(table)),
            })
    per_stock_counts = pd.DataFrame(count_rows)
    display(per_stock_counts.pivot(index="comparison", columns="loss",
                                   values=["reference significantly better", "reference significantly worse"]))
    per_stock_counts.to_csv(RESULTS_DIR / "Analysis" / "output" / "dm_per_stock_counts.csv", index=False)
    pd.concat(per_stock_all).to_csv(RESULTS_DIR / "Analysis" / "output" / "dm_per_stock.csv", index=False)

## Oracle and share of the attainable gain

The oracle uses, for each stock, the fixed lookback with the lowest full-sample QLIKE. It uses information from the test period, so it is an upper bound on what any lookback-selection rule could achieve, not a feasible forecast. `gain captured` is (QLIKE of LSTM_21 minus QLIKE of the rule) divided by (QLIKE of LSTM_21 minus QLIKE of the oracle), computed on the common sample; 100% means the rule matches the oracle, 0% means it does no better than LSTM_21.

In [ ]:
grid_models = [f"LSTM_{L}" for L in GRID]
rules = [m for m in ["LSTM_VAL", "LSTM_PLACEBO", "LSTM_AIC"] if m in predictions]
if all(m in predictions for m in grid_models) and rules:
    common_dates = actuals.index
    for m in grid_models + rules:
        common_dates = common_dates.intersection(predictions[m].index)
    tickers = sorted(set(actuals.columns).intersection(*(predictions[m].columns for m in grid_models + rules)))
    y = actuals.loc[common_dates, tickers].to_numpy(dtype=float)
    stack = {m: predictions[m].loc[common_dates, tickers].to_numpy(dtype=float) for m in grid_models + rules}
    valid = np.isfinite(y) & (y > 0)
    for a in stack.values():
        valid &= np.isfinite(a) & (a > 0)

    def stock_qlike(m):
        values = np.where(valid, loss_values(np.where(valid, y, 1.0), np.where(valid, stack[m], 1.0), "qlike"), np.nan)
        return pd.Series(np.nanmean(values, axis=0), index=tickers)

    per_stock_loss = pd.DataFrame({m: stock_qlike(m) for m in grid_models + rules})
    oracle_choice = per_stock_loss[grid_models].idxmin(axis=1)
    per_stock_loss["ORACLE"] = per_stock_loss[grid_models].min(axis=1)
    mean_loss = per_stock_loss.mean()
    gap = mean_loss["LSTM_21"] - mean_loss["ORACLE"]
    oracle_table = pd.DataFrame({
        "mean QLIKE": mean_loss,
        "gain captured vs LSTM_21": (mean_loss["LSTM_21"] - mean_loss) / gap if gap > 0 else np.nan,
    })
    display(oracle_table.style.format({"mean QLIKE": "{:.5f}", "gain captured vs LSTM_21": "{:.1%}"}))
    print("Oracle lookback by stock:", oracle_choice.value_counts().to_dict())
    oracle_table.to_csv(RESULTS_DIR / "Analysis" / "output" / "oracle_gain.csv")
    oracle_choice.rename("oracle_model").to_csv(RESULTS_DIR / "Analysis" / "output" / "oracle_choice.csv")
else:
    print("Oracle needs all of", grid_models, "and at least one of LSTM_VAL / LSTM_PLACEBO / LSTM_AIC.")